# POLARIS: U-Net SAR Oil-Spill Segmentation Training (Google Colab GPU)
### Probabilistic Maritime Pollution Attribution Engine

---

## Instructions for Running on Google Colab:
1. **Set Hardware Accelerator**: Go to **Runtime** → **Change runtime type** → select **T4 GPU** (or any available GPU).
2. **Upload Dataset to Google Drive**:
   - Upload your `archive` folder (or zip) containing `images/` and `masks/` to your Google Drive at:
     `MyDrive/polaris/archive/` (or update `DATA_DIR` in the config below).
3. **Run All Cells**: The notebook will:
   - Mount Google Drive.
   - Detect the GPU.
   - Train the U-Net model with SAR preprocessing, augmentation, and BCE+Dice loss.
   - Evaluate test-set metrics (IoU, Dice/F1, Precision, Recall).
   - Save `model.pth` and `metrics.json` directly back into your Google Drive at `MyDrive/polaris/models/`.
4. **Download Outputs**: Copy `model.pth` and `metrics.json` to your local repository's `data/models/` directory.

In [ ]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

# Check CUDA GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Go to Runtime -> Change runtime type -> T4 GPU for faster training.")

In [ ]:
import random
import json
import time
import cv2
import numpy as np
from PIL import Image
from typing import List, Tuple, Dict, Optional
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    # Adjust this path to where you placed the dataset in Google Drive
    "data_dir": Path("/content/drive/MyDrive/polaris/archive"),
    "model_out": Path("/content/drive/MyDrive/polaris/models/model.pth"),
    "metrics_out": Path("/content/drive/MyDrive/polaris/models/metrics.json"),
    
    "img_subdir": Path("images/images"),
    "mask_subdir": Path("masks/masks"),
    "subfolders": ["train", "val"],
    
    "num_classes": 2,       # 0=background, 1=oil-spill
    "img_size": 256,
    "train_frac": 0.70,
    "val_frac": 0.15,
    
    "epochs": 30,
    "batch_size": 16,       # Faster batch size on GPU
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "seed": 42,
    
    "aug_hflip": True,
    "aug_vflip": True,
    "aug_rotate": True,
    "aug_brightness": 0.15,
    "aug_contrast": 0.15,
    
    "bce_weight": 0.5,
    "dice_weight": 0.5,
    "early_stop_patience": 10,
}

CONFIG["model_out"].parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================================
# PREPROCESSING & ARCHITECTURE
# ============================================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def linear_to_db(sar_intensity: np.ndarray) -> np.ndarray:
    eps = 1e-7
    return 10.0 * np.log10(np.clip(sar_intensity.astype(np.float32), eps, None))

def lee_speckle_filter(image: np.ndarray, window_size: int = 5, damping_factor: float = 1.0) -> np.ndarray:
    if window_size % 2 == 0:
        window_size += 1
    img = image.astype(np.float32)
    mean = cv2.blur(img, (window_size, window_size))
    mean_sq = cv2.blur(img ** 2, (window_size, window_size))
    var = np.maximum(mean_sq - mean ** 2, 0.0)
    local_rv = var / np.maximum(mean ** 2, 1e-10)
    noise_var = 1.0 / 4.4
    w = np.clip(np.maximum(0.0, 1.0 - noise_var / np.maximum(local_rv, 1e-10)) * damping_factor, 0.0, 1.0)
    return mean + w * (img - mean)

def preprocess_sar_image(img_array: np.ndarray, target_size: int = 256) -> np.ndarray:
    if img_array.ndim == 3 and img_array.shape[2] == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    elif img_array.ndim == 3:
        gray = img_array[:, :, 0]
    else:
        gray = img_array.copy()
    db = linear_to_db(gray)
    filtered = lee_speckle_filter(db, window_size=5)
    if filtered.shape[0] != target_size or filtered.shape[1] != target_size:
        filtered = cv2.resize(filtered, (target_size, target_size), interpolation=cv2.INTER_AREA)
    p_lo, p_hi = np.percentile(filtered, (2, 98))
    norm = np.clip((filtered - p_lo) / max(p_hi - p_lo, 1e-3), 0.0, 1.0)
    return norm.astype(np.float32)

def preprocess_mask(mask_array: np.ndarray) -> np.ndarray:
    if mask_array.ndim == 3:
        mask_array = mask_array[:, :, 0]
    return (mask_array > 127).astype(np.int64)

class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class SimpleUNet(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.inc = DoubleConv(1, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(128, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(64, 32)
        self.outc = nn.Conv2d(32, num_classes, kernel_size=1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x = self.up1(x3)
        x = self.conv_up1(torch.cat([x, x2], dim=1))
        x = self.up2(x)
        x = self.conv_up2(torch.cat([x, x1], dim=1))
        return self.outc(x)

class DiceLoss(nn.Module):
    def __init__(self, num_classes: int, smooth: float = 1.0):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        one_hot = torch.zeros_like(probs)
        one_hot.scatter_(1, targets.unsqueeze(1), 1.0)
        dice_scores = []
        for c in range(1, self.num_classes):
            p = probs[:, c].reshape(-1)
            t = one_hot[:, c].reshape(-1)
            intersection = (p * t).sum()
            denom = p.sum() + t.sum() + self.smooth
            dice_scores.append(1.0 - (2.0 * intersection + self.smooth) / denom)
        return torch.stack(dice_scores).mean()

class CombinedLoss(nn.Module):
    def __init__(self, num_classes: int, bce_weight: float = 0.5, dice_weight: float = 0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss(num_classes)
        self.bce_w = bce_weight
        self.dice_w = dice_weight
    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        dice_loss = self.dice(logits, targets)
        combined = self.bce_w * ce_loss + self.dice_w * dice_loss
        if torch.isnan(combined):
            return ce_loss
        return combined

In [ ]:
# ============================================================================
# DATASET & METRICS DEFINITIONS
# ============================================================================

class OilSpillDataset(Dataset):
    def __init__(self, pairs, img_size: int, augment: bool = False, aug_cfg: Optional[Dict] = None):
        self.pairs = pairs
        self.img_size = img_size
        self.augment = augment
        self.aug_cfg = aug_cfg or {}
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img_arr = np.array(Image.open(img_path).convert("RGB"))
        mask_arr = np.array(Image.open(mask_path).convert("RGB"))
        img_proc = preprocess_sar_image(img_arr, self.img_size)
        mask_proc = preprocess_mask(mask_arr)
        if self.img_size != mask_proc.shape[0] or self.img_size != mask_proc.shape[1]:
            mask_proc = cv2.resize(mask_proc.astype(np.uint8), (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)
        img_t = torch.from_numpy(img_proc).unsqueeze(0)
        mask_t = torch.from_numpy(mask_proc)
        if self.augment:
            if self.aug_cfg.get("aug_hflip") and random.random() > 0.5:
                img_t = TF.hflip(img_t)
                mask_t = TF.hflip(mask_t.unsqueeze(0)).squeeze(0)
            if self.aug_cfg.get("aug_vflip") and random.random() > 0.5:
                img_t = TF.vflip(img_t)
                mask_t = TF.vflip(mask_t.unsqueeze(0)).squeeze(0)
            if self.aug_cfg.get("aug_rotate") and random.random() > 0.5:
                angle = random.uniform(-30.0, 30.0)
                img_t = TF.rotate(img_t, angle, interpolation=TF.InterpolationMode.BILINEAR)
                mask_t = TF.rotate(mask_t.unsqueeze(0), angle, interpolation=TF.InterpolationMode.NEAREST).squeeze(0)
            bmax = self.aug_cfg.get("aug_brightness", 0.0)
            if bmax > 0 and random.random() > 0.5:
                img_t = torch.clamp(img_t + random.uniform(-bmax, bmax), 0.0, 1.0)
            cmax = self.aug_cfg.get("aug_contrast", 0.0)
            if cmax > 0 and random.random() > 0.5:
                factor = random.uniform(1.0 - cmax, 1.0 + cmax)
                mean_val = img_t.mean()
                img_t = torch.clamp((img_t - mean_val) * factor + mean_val, 0.0, 1.0)
        return img_t, mask_t

def compute_metrics(preds: np.ndarray, targets: np.ndarray, oil_class: int = 1) -> Dict:
    tp = fp = fn = tn = 0
    for pred, gt in zip(preds, targets):
        p = (pred == oil_class)
        g = (gt == oil_class)
        tp += int(np.logical_and(p, g).sum())
        fp += int(np.logical_and(p, ~g).sum())
        fn += int(np.logical_and(~p, g).sum())
        tn += int(np.logical_and(~p, ~g).sum())
    eps = 1e-7
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    dice = 2 * tp / (2 * tp + fp + fn + eps)
    iou = tp / (tp + fp + fn + eps)
    return {
        "oil_iou": round(float(iou), 4),
        "oil_dice_f1": round(float(dice), 4),
        "oil_precision": round(float(precision), 4),
        "oil_recall": round(float(recall), 4),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

In [ ]:
# ============================================================================
# DATASET PREPARATION & TRAINING RUN
# ============================================================================
set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Device: {device}")

# Collect pairs
all_pairs = []
for sf in CONFIG["subfolders"]:
    img_dir = CONFIG["data_dir"] / CONFIG["img_subdir"] / sf
    mask_dir = CONFIG["data_dir"] / CONFIG["mask_subdir"] / sf
    for img_p in sorted(img_dir.glob("*.png")):
        mask_p = mask_dir / img_p.name
        if mask_p.exists():
            all_pairs.append((img_p, mask_p))

print(f"Found {len(all_pairs)} total pairs.")
rng = random.Random(CONFIG["seed"])
shuffled = list(all_pairs)
rng.shuffle(shuffled)

n = len(shuffled)
n_train = int(n * CONFIG["train_frac"])
n_val = int(n * CONFIG["val_frac"])
train_pairs = shuffled[:n_train]
val_pairs = shuffled[n_train:n_train + n_val]
test_pairs = shuffled[n_train + n_val:]
print(f"Split: {len(train_pairs)} train / {len(val_pairs)} val / {len(test_pairs)} test")

aug_cfg = {k: CONFIG[k] for k in CONFIG if k.startswith("aug_")}
train_ds = OilSpillDataset(train_pairs, CONFIG["img_size"], augment=True, aug_cfg=aug_cfg)
val_ds = OilSpillDataset(val_pairs, CONFIG["img_size"], augment=False)
test_ds = OilSpillDataset(test_pairs, CONFIG["img_size"], augment=False)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

model = SimpleUNet(num_classes=CONFIG["num_classes"]).to(device)
criterion = CombinedLoss(CONFIG["num_classes"], CONFIG["bce_weight"], CONFIG["dice_weight"])
optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

print("\nStarting Training Loop...")
best_val_iou = -1.0
best_epoch = -1
no_improve = 0

for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.time()
    model.train()
    total_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    train_loss = total_loss / len(train_loader.dataset)
    
    # Validation IoU
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs).argmax(dim=1)
            p = (preds == 1)
            g = (masks == 1)
            tp += int(torch.logical_and(p, g).sum().item())
            fp += int(torch.logical_and(p, ~g).sum().item())
            fn += int(torch.logical_and(~p, g).sum().item())
    val_iou = tp / (tp + fp + fn + 1e-7)
    scheduler.step(val_iou)
    elapsed = time.time() - t0
    lr_curr = optimizer.param_groups[0]["lr"]
    
    print(f"Epoch {epoch:2d}/{CONFIG['epochs']} | Loss: {train_loss:.5f} | Val IoU: {val_iou:.4f} | LR: {lr_curr:.2e} | Time: {elapsed:.1f}s")
    
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), CONFIG["model_out"])
        print(f"  --> Checkpoint saved (Best Val IoU: {best_val_iou:.4f})")
    else:
        no_improve += 1
        if no_improve >= CONFIG["early_stop_patience"]:
            print(f"Early stopping triggered after epoch {epoch}.")
            break

In [ ]:
# ============================================================================
# TEST EVALUATION & METRICS EXPORT
# ============================================================================
print("Evaluating Best Model on Held-out Test Set...")
model.load_state_dict(torch.load(CONFIG["model_out"], map_location=device))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for imgs, masks in test_loader:
        preds = model(imgs.to(device)).argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(masks.numpy())

test_metrics = compute_metrics(np.concatenate(all_preds), np.concatenate(all_targets), oil_class=1)
print("\n--- Test Set Metrics (Oil-Spill Class) ---")
for k, v in test_metrics.items():
    print(f"  {k:20s}: {v}")

full_metrics = {
    "model_info": {
        "num_classes": CONFIG["num_classes"],
        "img_size": CONFIG["img_size"],
        "architecture": "SimpleUNet (Google Colab GPU Training)",
        "note": "Interim binary model trained on Deep-SAR dataset. Switch num_classes=5 for multi-class."
    },
    "training": {
        "epochs_run": best_epoch,
        "best_val_iou": round(best_val_iou, 4),
        "seed": CONFIG["seed"],
        "train_pairs": len(train_pairs),
        "val_pairs": len(val_pairs),
        "test_pairs": len(test_pairs)
    },
    "test_metrics": test_metrics
}

with open(CONFIG["metrics_out"], "w") as f:
    json.dump(full_metrics, f, indent=2)

print(f"\nSaved metrics to: {CONFIG['metrics_out']}")
print(f"Saved model weights to: {CONFIG['model_out']}")